# pymatgen

## Example 1: computed entries and energies per atom

In [ ]:
# Example 1: ComputedEntry carries a composition and a total energy
from pymatgen.entries.computed_entries import ComputedEntry

entry = ComputedEntry("Li2O", -14.3, entry_id="li2o")
assert entry.composition.reduced_formula == "Li2O" and abs(entry.energy_per_atom - (-14.3 / 3)) < 1e-9
print("Example 1:", entry.entry_id, entry.composition, f"{entry.energy_per_atom:.3f} eV/atom")

## Example 2: a Li-O phase diagram

In [ ]:
# Example 2: PhaseDiagram finds Li2O stable and LiO2 above the hull for these made-up energies
from pymatgen.analysis.phase_diagram import PhaseDiagram
from pymatgen.entries.computed_entries import ComputedEntry

entries = [ComputedEntry("Li", -1.9), ComputedEntry("O2", -9.8), ComputedEntry("Li2O", -14.3), ComputedEntry("LiO2", -10.0)]
pd = PhaseDiagram(entries)
stable = sorted(e.composition.reduced_formula for e in pd.stable_entries)
assert "Li2O" in stable and "LiO2" not in stable, stable
print("Example 2: stable", stable, "| LiO2 above hull by", round(pd.get_e_above_hull(entries[3]), 3), "eV/atom")

## Example 3: an intercalation battery electrode

In [ ]:
# Example 3: InsertionElectrode from a delithiated and a lithiated entry gives a voltage
from pymatgen.apps.battery.insertion_battery import InsertionElectrode
from pymatgen.entries.computed_entries import ComputedEntry

li = ComputedEntry("Li", -1.9)
electrode = InsertionElectrode.from_entries([ComputedEntry("CoO2", -20.0), ComputedEntry("LiCoO2", -25.9)], li)
assert electrode.working_ion.symbol == "Li" and electrode.get_average_voltage() > 0
print("Example 3: average voltage", round(electrode.get_average_voltage(), 3), "V; capacity", round(electrode.get_capacity_grav(), 1), "mAh/g")

## Example 4: entries survive a JSON round trip

In [ ]:
# Example 4: MSONable as_dict/from_dict through monty's JSON encoder
import json
from monty.json import MontyDecoder, MontyEncoder
from pymatgen.entries.computed_entries import ComputedEntry

entry = ComputedEntry("Fe2O3", -38.0, parameters={"run_type": "GGA+U"})
again = json.loads(json.dumps(entry, cls=MontyEncoder), cls=MontyDecoder)
assert again.composition == entry.composition and again.energy == entry.energy and again.parameters["run_type"] == "GGA+U"
print("Example 4:", type(again).__name__, again.composition.reduced_formula, again.energy)

## Example 5: the `pmg` command

In [ ]:
# Example 5: the console script installed by this package answers --help
import shutil, subprocess

pmg = shutil.which("pmg")
assert pmg, "pmg is not on PATH"
run = subprocess.run([pmg, "--help"], capture_output=True, text=True, timeout=120)
assert run.returncode == 0 and "usage" in run.stdout.lower(), run.stderr[-300:]
print("Example 5:", pmg, "|", run.stdout.splitlines()[0][:80])